# Evaluation Pipeline

We will in this notebook setup the evaluation pipeline. We generated questions in ./generate_factuality_questions.ipynb that contain entries

- id
- question
- correct answer

Where we can sample 4 entries from annotated_dataset along with the incorrect entry, where the question pertains to. We will then evaluate the answer to the question by comparing it to the answer. We will use LLM-as-a-Judge and give it

- correct answer
- answer from LLM

and make it respond with a factuality score on a scale from 1 to 5.

In [ ]:
import os

import pandas as pd

path_to_questions_answers = "data/factuality_questions_answers.pkl"

if not os.path.exists(path_to_questions_answers):
    raise FileNotFoundError(
        "File does not exist. See ./generate_factuality_questions.ipynb to generate the file."
    )

df_questions = pd.read_pickle(path_to_questions_answers)
df_questions

In [ ]:
def sample_entries(df: pd.DataFrame, id: str, n: int = 5) -> pd.DataFrame:
    """Sample n entries from the dataframe with a specific id.

    Tries to be deterministic by setting by seeding the sampler (=22).

    Args:
        df (pd.DataFrame): The dataframe to sample from.
        id (str): The id to filter by (row entry always included in returned dataframe).
        n (int, optional): The number of samples to return. Defaults to 5. Must atleast be 1, as one entry with the given id is always included.

    Returns:
        pd.DataFrame: The sampled dataframe.
    """
    samples = df[df["id"] != id].sample(n - 1, random_state=22)
    samples = pd.concat([df[df["id"] == id], samples])
    return samples

In [ ]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = "smoldoc__en_sw"  # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset  # TODO we do not actually need the annotated version here, just the basic version

In [ ]:
annotated_df = pd.DataFrame(annotated_dataset)
annotated_df.head()

In [ ]:
from tqdm.notebook import tqdm
from llm_chat import CachedLLMChat, LLMChat, OllamaChatter, AzureOpenAIChatter
from pipeline import expose

verbose = False

# chatter = OllamaChatter(model_name="gemma3:4b")
# chatter = OllamaChatter(model_name="deepseek-r1:8b", think: True)
chatter = AzureOpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/evaluation_cache.pkl")

answers: list[dict[str, str]] = []

for id, question, ground_truth_answer, *_ in tqdm(
    df_questions.itertuples(index=False, name=None),
    total=len(df_questions),
    desc="Answering factuality questions",
):
    samples = sample_entries(annotated_df, id, n=5)

    if verbose:
        print(f"Question originates from document with ID: {id}")
        print(f"Question: {question}")
        print(f"Answer: {ground_truth_answer}")
        print(f"Using document samples with IDs: {' '.join(samples['id'])}")
        print("\n")

    expose(chat, samples)
    response, thoughts = chat.chat(question)
    if verbose:
        print(f"Q: {question}")
        if thoughts:
            print("*** THOUGHTS ***")
            print(thoughts)
            print("****************")
        print(f"A: {response}\n(Expected: {ground_truth_answer})\n{'-' * 80}\n")
    chat.reset()

    # collect answer and correct answer for evaluation later
    answers.append(
        {
            "id": id,
            "question": question,
            "expected answer": ground_truth_answer,
            "answer": response,
        }
    )


In [ ]:
answers_df = pd.DataFrame(answers)
answers_df

## Evaluating answers

In [ ]:
system_prompt = """\
You are an expert evaluator that will score answers based on their factual accuracy.

The answer you provide should be an integer from 1 to 5, where:
1 - The answer is completely incorrect or irrelevant.
2 - The answer has significant inaccuracies or omissions.
3 - The answer is partially correct but lacks important details.
4 - The answer is mostly correct with minor inaccuracies.
5 - The answer is completely correct and comprehensive.

Remember: Your answer should only be an integer between 1 and 5, and nothing else!
"""

chatter = OllamaChatter(model_name="gemma3:4b")
# chatter = AzureOpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/evaluation_cache.pkl")

scores: list[int] = []

for id, q, expected, a in tqdm(answers_df.itertuples(index=False, name=None), total=len(answers_df), desc="Scoring answers"):
    chat.add_message("system", system_prompt)
    response, _ = chat.chat(f"Question: {q}\nExpected answer: {expected}\nAnswer given: {a}\n\nPlease provide a score from 1 to 5.")
    print(f"Question: {q}")
    print(f"Expected answer: {expected}")
    print(f"Answer given: {a}")
    print(f"ID: {id}\nScore: {response}\n{'-'*80}\n")
    scores.append(int(response))
    chat.reset()

In [ ]:
average_score = sum(scores) / len(scores)
print(f"Average factuality score: {average_score:.2f} out of 5")